In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,List,Sequence,Annotated


In [ ]:
class EmployeeState(TypedDict):
    name: str
    salary: float
    age: int
    decision: Literal["std_hr","forced_hr"]
    result:str

In [ ]:
def analyze_employee(state :EmployeeState)->EmployeeState:
    age=state["age"]
    salary=state['salary']
    if age<=30 and salary<=40000:
        state["decision"]="std_hr"
    else:       
        state["decision"]="forced_hr"
        print(f"le dosssier de {state['name']}a ete analysee")
    return state

In [ ]:
def standard_hr_process(state:EmployeeState):
    state['result']= f"""demande {state['name']}
     envoyee au traitement RH standard"""
    print("*"*10)
    print(f"le dosssier de {state['name']}a ete traite par le processus RH standard")
    return state

In [ ]:
def foced_hr_process(state:EmployeeState):
    state['result']=f"""demande de {state['name']}
    envoyee a la validation RH forcee"""
    print("*"*10)
    print(f"le dosssier de {state['name']}a ete traite par le processus RH forcee")
    return state

In [ ]:
def router (state:EmployeeState):
    return state['decision']

In [ ]:
workflow=StateGraph(EmployeeState)
workflow.add_node("analyze_node",analyze_employee)
workflow.add_node("standard_hr_node",standard_hr_process)
workflow.add_node("forced_hr_node",foced_hr_process)
workflow.add_edge(START ,"analyze_node")
workflow.add_conditional_edges(
    "analyze_node",
    router,
    {
        "std_hr":"standard_hr_node",
        "forced_hr":"forced_hr_node"
    }
)
workflow.add_edge("standard_hr_node",END)
workflow.add_edge("forced_hr_node",END)
graph=workflow.compile()



In [ ]:
from IPython.display import Image 

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result=graph.invoke({
    "name":"Alice",
    "age": 30,
    "salary": 20000
    
})

In [ ]:
result=graph.invoke({
    "name":"hanna",
    "age": 30,
    "salary": 70000
})

In [ ]:
from langchain.tools import tool
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from langchain_core.messages import BaseMessage,AnyMessage
from langgraph.graph import add_messages
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode
from dotenv.ipython import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
load_dotenv(override=True)

In [ ]:
@tool
def add(a:float,b:float):
    """ArithmeticError: adds two numbers a and b and returns the result"""
    print(f"adding {a} and {b}")
    return a+b
@tool
def muliply (a:float,b:float):
    """ArithmeticError: adds two numbers a and b and returns the result"""
    print(f"Multiplying {a} and {b}")
    return a*b 
@tool
def divide (a:float,b:float):
    """ArithmeticError: adds two numbers a and b and returns the result"""
    print(f"dividing {a} by {b}")
    return a/b
tools=[add,muliply,divide]

In [ ]:
class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],add_messages]

In [ ]:
llm =ChatOpenAI(model="gpt-4o",temperature=0)
llm_with_tools=llm.bind_tools(tools=tools)

In [ ]:
def assistant(state:AgentState)->AgentState:
     response =llm_with_tools.invoke(state["messages"])
     print(f"llm invoked")
     return{"messages":[response]}


In [ ]:
def should_continue(state:AgentState)->bool:
    last_message=state["messages"][-1]
    if not last_message.tool_calls:
        return "end"
    else:      
         return "continue"

In [ ]:
workflow=StateGraph(AgentState)
workflow.add_node("assistant",assistant)
workflow.add_node("tools",ToolNode(tools =tools))
workflow.set_entry_point("assistant")
workflow.add_conditional_edges(
    "assistant",
    should_continue,
    {
      "continue": "tools",
      "end": END
    }
    )
workflow.add_edge("tools","assistant")
memory=InMemorySaver()
graph=workflow.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable":{"thread_id": 1}}
resp =graph.invoke({
    "messages":[HumanMessage(content="what is the sum of 2 and 20")]
},config=config)

In [ ]:
print(resp)

In [ ]:
resp =graph.invoke({
    "messages":[HumanMessage("bonjour je m'apppelle marwa")]
}, config=config) 
print(resp["messages"][-1].content)


In [ ]:
resp =graph.invoke({
    "messages":[HumanMessage("comment je m'apppelle ")]
}, config=config) 
print(resp["messages"][-1].content)